In [41]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import sys
import os

home = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

sys.path.append(home)

In [ ]:
catalog = dbutils.widgets.get("catalog")
checkpoints_dir = dbutils.widgets.get("checkpoints_dir")

In [ ]:
df = spark.readStream.table(f"{catalog}.brz.employee_cdc")

# the below condition is called bootstrap load, its not a guarantee that every firm may be tracking historical dimensional
# changes since inception, hence, during first load it becomes tricky while implementing SCD2 style table for a data that
# has been existing since a while, as if we track the recent operation per each employee for every batch, we will definitely miss
# all the history, as only recent records will be populated. This is the reason why we follow bootstrap load the first load when implementing
# a data platform, so all insert records or all the earliest entrie per employee would be inserted once only when the table is empty,
# if not emmpty, we can follow our generic SCD2 merge upsert method.

def scd_imp(batch_df, batch_id):

        rows = spark.sql(f"select surrogate_key from {catalog}.brz.dim_employee_historical limit 1").collect()

        if len(rows) == 0:
                
                w = Window.partitionBy("employee_id").orderBy(F.col("event_time"))

                insert_df = (batch_df.filter(F.col("operation")=="I")
                                .withColumn("rn", F.row_number().over(w))
                                .filter(F.col("rn")==1)
                                .drop("rn"))

                insert_df = (insert_df.select(F.sha2(
                                                F.concat_ws("||", 
                                                                F.col("employee_id").cast("string"),
                                                                F.col("event_time").cast("string"),
                                                                F.col("operation").cast("string"))
                                                        ,256).alias("surrogate_key"),
                                                "employee_id", F.col("after_employee_name").alias("employee_name"),
                                                F.col("after_role").alias("role"), F.col("after_department").alias("department"),
                                                F.col("after_region_id").alias("region_id"), F.col("after_joining_date").alias("joining_date"),
                                                F.col("after_salary").alias("salary"), F.to_date(F.col("event_time")).alias("start_date"),
                                                F.lit(None).cast("date").alias("end_date"), F.lit(True).alias("is_curr"), F.lit(False).alias("is_del")))
                
                insert_df.write.format("delta").mode("append").saveAsTable(f"{catalog}.brz.dim_employee_historical")
        else:
                print("table populated already, implementing SCD2 logic")

                w = Window.partitionBy("employee_id").orderBy(F.col("event_time").desc())

                df = (batch_df.withColumn("rn", F.row_number().over(w))
                        .filter(F.col("rn")==1)
                        .drop("rn"))

                target = DeltaTable.forName(spark, f"{catalog}.brz.dim_employee_historical")
        
        # merge upsert to implement scd2
        
                target.alias("t").merge(df.alias("s"),
                                        "s.employee_id = t.employee_id AND t.is_curr = true")\
                .whenMatchedUpdate( # for 'U'/updated operation, expire current row
                        condition = "s.operation = 'U'",
                        set = {
                                "end_date": "to_date(s.event_time)",
                                "is_curr": "false"
                        }
                )\
                .whenMatchedUpdate( # for 'D'/deleted operation, expire current row and set to delete true
                        condition = "s.operation = 'D'",
                        set = {
                                "end_date": "to_date(s.event_time)",
                                "is_curr": "false",
                                "is_del": "true"                        
                        }
                ).execute()

                # to insert updated and inserted records as new append records

                new_rows = (df.filter(F.col("operation").isin(["U", "I"]))
                                .select(F.sha2(
                                                F.concat_ws("||", 
                                                                F.col("employee_id").cast("string"),
                                                                F.col("event_time").cast("string"),
                                                                F.col("operation").cast("string"))
                                                        ,256).alias("surrogate_key"),
                                                "employee_id", F.col("after_employee_name").alias("employee_name"),
                                                F.col("after_role").alias("role"), F.col("after_department").alias("department"),
                                                F.col("after_region_id").alias("region_id"), F.col("after_joining_date").alias("joining_date"),
                                                F.col("after_salary").alias("salary"), F.to_date(F.col("event_time")).alias("start_date"),
                                                F.lit(None).cast("date").alias("end_date"), F.lit(True).alias("is_curr"), F.lit(False).alias("is_del")))

                new_rows.write.format("delta").mode("append").saveAsTable(f"{catalog}.brz.dim_employee_historical")

query = (df.writeStream
                .foreachBatch(scd_imp)
                .option("checkpointLocation", f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/brz_checkpoints/employee_scd_hist_checkpoint")
                .trigger(availableNow = True)
                .start())

query.awaitTermination(300)

True